# Inventário dos cursos


In [18]:
import pandas as pd
from pathlib import Path

pd.set_option("display.max_rows", 500)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
from pathlib import Path


def resolver_base() -> Path:
    """Resolve o diretório raiz do projeto em qualquer ambiente."""
    cwd = Path.cwd().resolve()

    for candidate in [cwd, *cwd.parents]:
        if (candidate / "tsv").exists() and (candidate / "catalogos").exists():
            return candidate

    return cwd


BASE = resolver_base()

DIR_TSV = BASE / "tsv"
DIR_CATALOGOS = BASE / "catalogos"
DIR_RELATORIOS = BASE / "relatorios"

DIR_CATALOGOS.mkdir(parents=True, exist_ok=True)
DIR_RELATORIOS.mkdir(parents=True, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [20]:
TRIENIOS = [
    "2018-2020",
    "2019-2021",
    "2020-2022",
    "2021-2023",
    "2022-2024",
    "2023-2025",
]

ARQUIVOS = {
    trienio: DIR_TSV / f"{trienio}.tsv"
    for trienio in TRIENIOS
}

In [21]:
def carregar_tsv(arquivo):
    """
    Lê um TSV do PAS detectando automaticamente
    a linha do cabeçalho.
    """

    bruto = pd.read_csv(
        arquivo,
        sep="\t",
        header=None,
        dtype=str,
        keep_default_na=False,
    )

    linha = None

    for i, valores in bruto.iterrows():

        valores = (
            valores.fillna("")
                   .astype(str)
                   .str.strip()
                   .tolist()
        )

        if {"Curso", "Turno"}.issubset(valores):
            linha = i
            break

    if linha is None:
        raise ValueError(
            f"Cabeçalho não encontrado em {arquivo.name}"
        )

    df = pd.read_csv(
        arquivo,
        sep="\t",
        header=linha,
        dtype=str,
        keep_default_na=False,
    )

    df.columns = (
        pd.Index(df.columns)
        .astype(str)
        .str.strip()
    )

    return df


def carregar_dados(arquivos):
    """
    Carrega todos os TSVs do projeto.
    """

    dados = {}

    print("=" * 70)
    print("CARREGANDO TSVs")
    print("=" * 70)

    for trienio, arquivo in arquivos.items():

        if not arquivo.exists():
            raise FileNotFoundError(arquivo)

        df = carregar_tsv(arquivo)

        obrigatorias = {"Curso", "Turno"}

        faltantes = obrigatorias - set(df.columns)

        if faltantes:
            raise ValueError(
                f"{arquivo.name}: colunas ausentes {sorted(faltantes)}"
            )

        dados[trienio] = df

        print(
            f"✓ {trienio:<10}"
            f" {len(df):>4} linhas"
            f" | {len(df.columns):>2} colunas"
        )

    return dados

In [22]:
DADOS = carregar_dados(ARQUIVOS)

CARREGANDO TSVs
✓ 2018-2020   109 linhas | 22 colunas
✓ 2019-2021   100 linhas | 23 colunas
✓ 2020-2022   106 linhas | 22 colunas
✓ 2021-2023   105 linhas | 22 colunas
✓ 2022-2024    60 linhas | 22 colunas
✓ 2023-2025   103 linhas | 22 colunas


# Normalização do Inventário

In [23]:
# ============================================================
# HISTÓRICO DOS CURSOS
# ============================================================

historico = []

for trienio, df in DADOS.items():

    cursos = (
        df["Curso"]
        .fillna("")
        .astype(str)
        .str.strip()
    )

    cursos = cursos[cursos != ""]

    historico.extend(
        {
            "Triênio": trienio,
            "Curso": curso
        }
        for curso in cursos
    )

historico = pd.DataFrame(historico)

display(historico.head())

,Triênio,Curso
0,2018-2020,Campus UnB — Ceilândia / DF
1,2018-2020,Terapia Ocupacional (Bacharelado)
2,2018-2020,Fonoaudiologia (Bacharelado)
3,2018-2020,Fisioterapia (Bacharelado)
4,2018-2020,Farmácia (Bacharelado)


In [24]:
# ============================================================
# FREQUÊNCIAS
# ============================================================

frequencias = (

    historico

    .groupby("Curso")

    .agg(

        Ocorrências=("Curso", "size"),

        Triênios=("Triênio", "nunique"),

        Lista_de_triênios=(

            "Triênio",

            lambda x: ", ".join(sorted(set(x)))

        )

    )

    .reset_index()

    .sort_values(

        [

            "Triênios",

            "Ocorrências",

            "Curso"

        ],

        ascending=[False, False, True]

    )

)

display(frequencias)

,Curso,Ocorrências,Triênios,Lista_de_triênios
60,Farmácia (Bacharelado),18,6,"2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025"
0,Administração (Bacharelado),12,6,"2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025"
2,Arquitetura e Urbanismo (Bacharelado),12,6,"2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025"
41,Direito (Bacharelado),12,6,"2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025"
46,Enfermagem (Bacharelado),12,6,"2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025"
122,Pedagogia (Licenciatura),11,6,"2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025"
27,Ciências Contábeis (Bacharelado),10,6,"2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025"
1,Agronomia (Bacharelado),6,6,"2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025"
14,Biotecnologia (Bacharelado),6,6,"2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025"
19,Ciência Política (Bacharelado),6,6,"2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025"


In [25]:
# ============================================================
# MATRIZ HISTÓRICA
# ============================================================

matriz = (

    historico

    .assign(Presente=1)

    .pivot_table(

        index="Curso",

        columns="Triênio",

        values="Presente",

        aggfunc="max",

        fill_value=0

    )

    .astype(int)

)

display(matriz)

Triênio,2018-2020,2019-2021,2020-2022,2021-2023,2022-2024,2023-2025
Curso,,,,,,
Administração (Bacharelado),1,1,1,1,1,1
Agronomia (Bacharelado),1,1,1,1,1,1
Arquitetura e Urbanismo (Bacharelado),1,1,1,1,1,1
Arquivologia,0,1,0,0,0,0
Arquivologia (Bacharelado),1,0,1,1,0,1
Artes Cênicas (Bacharelado),0,0,0,0,0,1
Artes Cênicas (Licenciatura),0,0,0,0,0,1
Artes Cênicas - Interpretação Teatral (Bacharelado),0,0,0,1,1,0
Artes Cênicas - Interpretação Teatral (Bacharelado) *,1,1,1,0,0,0


In [26]:
# ============================================================
# ESTATÍSTICAS
# ============================================================

todos = matriz.index[
    matriz.sum(axis=1) == len(DADOS)
]

exclusivos = matriz.index[
    matriz.sum(axis=1) == 1
]

print("=" * 70)
print("ESTATÍSTICAS")
print("=" * 70)

print(f"Cursos distintos        : {len(frequencias)}")
print(f"Presentes em todos      : {len(todos)}")
print(f"Exclusivos de triênio   : {len(exclusivos)}")

ESTATÍSTICAS
Cursos distintos        : 135
Presentes em todos      : 34
Exclusivos de triênio   : 40


In [27]:
# ============================================================
# RELATÓRIOS
# ============================================================

print()

print("CURSOS PRESENTES EM TODOS OS TRIÊNIOS")

print("-" * 70)

for curso in sorted(todos):
    print(curso)

print()

print("CURSOS EXCLUSIVOS")

print("-" * 70)

for curso in sorted(exclusivos):
    print(curso)

print()

print("CURSOS POR TRIÊNIO")

print("-" * 70)

for trienio, df in DADOS.items():

    quantidade = historico.loc[
        historico["Triênio"] == trienio,
        "Curso"
    ].nunique()

    print(f"{trienio}: {quantidade}")


CURSOS PRESENTES EM TODOS OS TRIÊNIOS
----------------------------------------------------------------------
Administração (Bacharelado)
Agronomia (Bacharelado)
Arquitetura e Urbanismo (Bacharelado)
Biotecnologia (Bacharelado)
Ciência Política (Bacharelado)
Ciência da Computação (Bacharelado)
Ciências Contábeis (Bacharelado)
Ciências Econômicas (Bacharelado)
Computação (Licenciatura)
Comunicação Organizacional (Bacharelado)
Comunicação Social – Audiovisual (Bacharelado)
Direito (Bacharelado)
Enfermagem (Bacharelado)
Engenharia Civil (Bacharelado)
Engenharia Elétrica (Bacharelado)
Engenharia Mecânica (Bacharelado)
Engenharia Química (Bacharelado)
Engenharia de Computação (Bacharelado)
Engenharia de Produção (Bacharelado)
Engenharia de Redes de Comunicação (Bacharelado)
Engenharias – Aeroespacial / Automotiva / Eletrônica / Energia / Software (Bacharelados)**
Estatística (Bacharelado)
Farmácia (Bacharelado)
Fisioterapia (Bacharelado)
Fonoaudiologia (Bacharelado)
Jornalismo (Bacharelado)

In [28]:
# ============================================================
# EXPORTAÇÃO
# ============================================================

frequencias.to_csv(
    DIR_RELATORIOS / "frequencias_cursos.csv",
    index=False,
    encoding="utf-8-sig"
)

matriz.to_csv(
    DIR_RELATORIOS / "matriz_historica_cursos.csv",
    encoding="utf-8-sig"
)

historico.to_csv(
    DIR_RELATORIOS / "historico_cursos.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Arquivos exportados com sucesso.")

Arquivos exportados com sucesso.


# Relatório para avaliação humana

In [29]:
# ============================================================
# PREPARAÇÃO
# ============================================================

import re


def simplificar_nome(nome):
    """
    Produz uma versão simplificada apenas para comparação.
    Não altera o nome original.
    """

    nome = nome.lower()

    nome = re.sub(r"\(.*?\)", "", nome)

    nome = nome.replace("–", "-")
    nome = nome.replace("—", "-")

    nome = re.sub(r"\s+", " ", nome)

    return nome.strip()


frequencias["Nome simplificado"] = (
    frequencias["Curso"]
    .apply(simplificar_nome)
)

In [30]:
# ============================================================
# AGRUPAMENTOS IMEDIATOS
# ============================================================

agrupamentos = (

    frequencias

    .groupby("Nome simplificado")

    .agg(

        Cursos=("Curso", list),

        Quantidade=("Curso", "size"),

        Ocorrências=("Ocorrências", "sum")

    )

    .reset_index()

)

agrupamentos = agrupamentos[
    agrupamentos["Quantidade"] > 1
]

display(agrupamentos)

,Nome simplificado,Cursos,Quantidade,Ocorrências
3,arquivologia,"[Arquivologia (Bacharelado), Arquivologia]",2,5
4,artes cênicas,"[Artes Cênicas (Licenciatura), Artes Cênicas (Bacharelado)]",2,3
8,artes visuais *,"[Artes Visuais (Bacharelado)*, Artes Visuais (Licenciatura)*]",2,7
18,ciências ambientais,"[Ciências Ambientais (Bacharelado), Ciências Ambientais]",2,5
19,ciências biológicas,"[Ciências Biológicas (Bacharelado), Ciências Biológicas, Ciências Biológicas (Licenciatura)]",3,7
20,ciências contábeis,"[Ciências Contábeis (Bacharelado), Ciências Contábeis]",2,11
23,ciências sociais - antropologia / sociologia,"[Ciências Sociais – Antropologia / Sociologia (Bacharelado/ Licenciatura), Ciências Sociais – Antropologia / Sociologia (Bacharelado/Licenciat ura)]",2,4
28,comunicação social - publicidade e propaganda,"[Comunicação Social - Publicidade e Propaganda (Bacharelado), Comunicação Social – Publicidade e Propaganda (Bacharelado)]",2,6
33,educação física,"[Educação Física (Bacharelado), Educação Física (Licenciatura)]",2,8
34,educação física ciclo básico,"[Educação Física Ciclo Básico, Educação Física Ciclo Básico (Bacharelado/Licenciatura)]",2,2


In [31]:
# ============================================================
# SIMILARIDADE
# ============================================================

!pip -q install rapidfuzz

from rapidfuzz import fuzz

pares = []

nomes = frequencias["Curso"].tolist()

for i, nome1 in enumerate(nomes):

    for nome2 in nomes[i + 1:]:

        score = max(

            fuzz.ratio(
                simplificar_nome(nome1),
                simplificar_nome(nome2)
            ),

            fuzz.token_sort_ratio(
                simplificar_nome(nome1),
                simplificar_nome(nome2)
            ),

            fuzz.token_set_ratio(
                simplificar_nome(nome1),
                simplificar_nome(nome2)
            )

        )

        if score >= 90:

            pares.append({

                "Curso A": nome1,

                "Curso B": nome2,

                "Similaridade": score

            })

pares = (

    pd.DataFrame(pares)

    .sort_values(

        "Similaridade",

        ascending=False

    )

)

display(pares)

,Curso A,Curso B,Similaridade
0,Ciências Contábeis (Bacharelado),Ciências Contábeis,100.000000
1,Ciência da Computação (Bacharelado),Computação (Licenciatura),100.000000
2,Computação (Licenciatura),Engenharia de Computação (Bacharelado),100.000000
3,Engenharia Química (Bacharelado),Química (Bacharelado),100.000000
4,Engenharia Química (Bacharelado),Química (Licenciatura),100.000000
5,Medicina (Bacharelado),Medicina Veterinária (Bacharelado),100.000000
6,Química (Bacharelado),Química Tecnológica (Bacharelado),100.000000
7,Química (Bacharelado),Licenciatura em Química,100.000000
8,Química (Bacharelado),Química (Licenciatura),100.000000
9,Ciências Biológicas (Bacharelado),Licenciatura em Ciências Biológicas,100.000000


In [32]:
# ============================================================
# CANDIDATOS
# ============================================================

def trienios(curso):

    return sorted(

        historico.loc[

            historico["Curso"] == curso,

            "Triênio"

        ].unique()

    )


candidatos = []

for _, linha in pares.iterrows():

    a = linha["Curso A"]
    b = linha["Curso B"]

    candidatos.append({

        "Curso A": a,

        "Curso B": b,

        "Similaridade": linha["Similaridade"],

        "Triênios A": ", ".join(trienios(a)),

        "Triênios B": ", ".join(trienios(b)),

        "Ocorrências A": int(

            frequencias.loc[
                frequencias["Curso"] == a,
                "Ocorrências"
            ].iloc[0]
        ),

        "Ocorrências B": int(

            frequencias.loc[
                frequencias["Curso"] == b,
                "Ocorrências"
            ].iloc[0]
        )

    })

candidatos = pd.DataFrame(candidatos)

display(candidatos)

,Curso A,Curso B,Similaridade,Triênios A,Triênios B,Ocorrências A,Ocorrências B
0,Ciências Contábeis (Bacharelado),Ciências Contábeis,100.000000,"2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025",2019-2021,10,1
1,Ciência da Computação (Bacharelado),Computação (Licenciatura),100.000000,"2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025","2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025",6,6
2,Computação (Licenciatura),Engenharia de Computação (Bacharelado),100.000000,"2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025","2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025",6,6
3,Engenharia Química (Bacharelado),Química (Bacharelado),100.000000,"2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025","2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025",6,6
4,Engenharia Química (Bacharelado),Química (Licenciatura),100.000000,"2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025",2023-2025,6,1
5,Medicina (Bacharelado),Medicina Veterinária (Bacharelado),100.000000,"2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025","2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025",6,6
6,Química (Bacharelado),Química Tecnológica (Bacharelado),100.000000,"2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025","2018-2020, 2019-2021, 2020-2022, 2021-2023, 2023-2025",6,5
7,Química (Bacharelado),Licenciatura em Química,100.000000,"2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025","2018-2020, 2019-2021, 2020-2022, 2021-2023",6,4
8,Química (Bacharelado),Química (Licenciatura),100.000000,"2018-2020, 2019-2021, 2020-2022, 2021-2023, 2022-2024, 2023-2025",2023-2025,6,1
9,Ciências Biológicas (Bacharelado),Licenciatura em Ciências Biológicas,100.000000,"2018-2020, 2020-2022, 2021-2023, 2022-2024, 2023-2025","2018-2020, 2019-2021, 2020-2022, 2021-2023",5,4


In [33]:
# ============================================================
# EXPORTAÇÃO
# ============================================================

candidatos.to_csv(
    DIR_RELATORIOS / "candidatos_normalizacao.csv",
    index=False,
    encoding="utf-8-sig"
)

agrupamentos.to_csv(
    DIR_RELATORIOS / "agrupamentos_imediatos.csv",
    index=False,
    encoding="utf-8-sig"
)

print("=" * 70)
print("RELATÓRIOS GERADOS")
print("=" * 70)

print("✓ candidatos_normalizacao.csv")
print("✓ agrupamentos_imediatos.csv")

RELATÓRIOS GERADOS
✓ candidatos_normalizacao.csv
✓ agrupamentos_imediatos.csv


In [34]:
# ============================================================
# PLANILHA DE REVISÃO
# ============================================================

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill

wb = Workbook()
ws = wb.active
ws.title = "Cursos"

cabecalho = [
    "Curso histórico",
    "Ocorrências",
    "Nº triênios",
    "Lista de triênios",
    "Nome simplificado",
    "Curso canônico",
    "Ação",
    "Observação",
]

for coluna, texto in enumerate(cabecalho, start=1):

    celula = ws.cell(row=1, column=coluna)

    celula.value = texto
    celula.font = Font(bold=True)
    celula.fill = PatternFill(
        fill_type="solid",
        fgColor="D9EAD3"
    )

for _, linha in frequencias.iterrows():

    ws.append([

        linha["Curso"],

        int(linha["Ocorrências"]),

        int(linha["Triênios"]),

        linha["Lista_de_triênios"],

        linha["Nome simplificado"],

        "",

        "",

        ""

    ])

# Ajuste simples da largura das colunas

larguras = {
    "A": 55,
    "B": 12,
    "C": 12,
    "D": 30,
    "E": 55,
    "F": 55,
    "G": 18,
    "H": 40,
}

for coluna, largura in larguras.items():
    ws.column_dimensions[coluna].width = largura

arquivo = DIR_CATALOGOS / "revisao_cursos.xlsx"

wb.save(arquivo)

print("=" * 70)
print("PLANILHA DE REVISÃO GERADA")
print("=" * 70)
print(arquivo)

PLANILHA DE REVISÃO GERADA
/content/drive/MyDrive/Dados/PAS/catalogos/revisao_cursos.xlsx
